# Day 035 — Exercise 4: generate_digest

**What you'll build:** `generate_digest(results, model) -> str` — filter the batch results to successful ones, format a numbered context block, and call the LLM once to write a 3-4 sentence editorial digest.

**Why it matters:** The delivery layer. All the extraction work feeds into one synthesis call that produces a human-readable output — the digest a user would actually read each morning.

## Provided: All Functions up to batch_extract

In [ ]:
import requests
from pathlib import Path

def fetch_text(source: str) -> dict:
    src  = str(source)
    text = None
    kind = 'text'

    if src.startswith('http://') or src.startswith('https://'):
        try:
            response = requests.get(src, timeout=10)
            response.raise_for_status()
            text = response.text
            kind = 'url'
        except Exception as e:
            text = '[fetch error: ' + str(e) + ']'
            kind = 'url_error'
    else:
        try:
            p = Path(src)
            if p.exists() and p.is_file():
                text = p.read_text(encoding='utf-8')
                kind = 'file'
        except Exception:
            pass

    if text is None:
        text = src
        kind = 'text'

    return {
        'source':     src,
        'kind':       kind,
        'content':    text,
        'char_count': len(text),
    }


import json
import ollama
from pydantic import BaseModel, Field

class ArticleInfo(BaseModel):
    title:      str       = Field(description='Topic or title in 3-6 words')
    summary:    str       = Field(description='One sentence summary')
    sentiment:  str       = Field(description='positive, negative, or neutral')
    key_points: list[str] = Field(default_factory=list,
                                  description='Up to 3 key points as short phrases')

def extract_info(doc: dict, model: str = 'llama3.2') -> dict:
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}


import asyncio
import json
import ollama

async def async_extract(doc: dict, model: str = 'llama3.2') -> dict:
    client = ollama.AsyncClient()
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = await client.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}


async def batch_extract(docs: list, max_concurrent: int = 3,
                        model: str = 'llama3.2') -> list[dict]:
    if not docs:
        return []
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(doc):
        async with sem:
            return await async_extract(doc, model)
    return list(await asyncio.gather(*[_run(d) for d in docs]))

## Provided: Sample Results (for checks)

In [ ]:
SAMPLE_RESULTS = [
    {
        'source': 'python.txt', 'kind': 'text',
        'content': 'Python is a popular language.', 'char_count': 29,
        'info': {
            'title': 'Python programming language',
            'summary': 'Python is widely used in data science and AI.',
            'sentiment': 'positive',
            'key_points': ['high-level syntax', 'large ecosystem', 'used in AI'],
        },
        'status': 'ok', 'error': None,
    },
    {
        'source': 'ml.txt', 'kind': 'text',
        'content': 'ML uses stats.', 'char_count': 15,
        'info': {
            'title': 'Machine learning basics',
            'summary': 'Machine learning finds patterns in data using statistics.',
            'sentiment': 'neutral',
            'key_points': ['statistical methods', 'pattern recognition'],
        },
        'status': 'ok', 'error': None,
    },
    {
        'source': 'bad.txt', 'kind': 'text',
        'content': 'bad', 'char_count': 3,
        'info': None, 'status': 'error', 'error': 'validation failed',
    },
]

## Your Implementation

In [ ]:
import ollama

def generate_digest(results: list, model: str = 'llama3.2') -> str:
    """
    Synthesise a digest from batch_extract results.

    Filter ok results, format a numbered context block, call ollama.chat once,
    return the editorial digest string.
    """
    ok     = [r for r in results if r.get('status') == 'ok']
    errors = [r for r in results if r.get('status') == 'error']

    # TODO: if not ok: return 'No articles extracted successfully (N errors).'

    # TODO: Build lines list:
    #   header line: '=== Auto-Analyst Digest ==='
    #   count line: 'N sources processed: M ok, K failed.'
    #   for i, r in enumerate(ok, 1):
    #       info = r.get('info') or {}
    #       title, summary, sentiment, kp = info.get(...)
    #       lines: '[i] title [sentiment]', '    summary', '    Key points: ...' (if any)
    #       append '' for blank line between entries

    # TODO: context = '\n'.join(lines)
    # TODO: prompt = context + '\n\nWrite a 3-4 sentence editorial digest...'
    # TODO: response = ollama.chat(model=model, messages=[...])
    # TODO: return response['message']['content']
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'generate_digest' in globals()
        passed += 1; print('\u2705 Check 1: generate_digest defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: empty input returns non-empty string (early return)
    try:
        r0 = generate_digest([])
        assert isinstance(r0, str) and r0.strip(), \
            'generate_digest([]) should return a non-empty string'
        passed += 1; print(f'\u2705 Check 2: generate_digest([]) returns str: {r0!r}')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: all-error input returns without LLM call
    try:
        err_results = [r for r in SAMPLE_RESULTS if r['status'] == 'error']
        r_err = generate_digest(err_results)
        assert isinstance(r_err, str) and r_err.strip()
        passed += 1; print(f'\u2705 Check 3: all-error input returns str without LLM call')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: SAMPLE_RESULTS returns non-empty string (1 LLM call)
    try:
        digest = generate_digest(SAMPLE_RESULTS)
        assert isinstance(digest, str), f'expected str, got {type(digest).__name__}'
        assert len(digest.strip()) >= 50, \
            f'digest too short: {len(digest)} chars'
        passed += 1; print(f'\u2705 Check 4: returns substantive digest ({len(digest)} chars)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: handles mixed ok/error without crashing
    try:
        r5 = generate_digest(SAMPLE_RESULTS)
        assert isinstance(r5, str) and r5.strip()
        passed += 1; print('\u2705 Check 5: mixed ok/error results handled gracefully')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import ollama

def generate_digest(results: list, model: str = 'llama3.2') -> str:
    ok     = [r for r in results if r.get('status') == 'ok']
    errors = [r for r in results if r.get('status') == 'error']
    if not ok:
        return 'No articles extracted successfully (' + str(len(errors)) + ' errors).'
    lines = [
        '=== Auto-Analyst Digest ===',
        str(len(results)) + ' sources processed: '
        + str(len(ok)) + ' ok, ' + str(len(errors)) + ' failed.\n',
    ]
    for i, r in enumerate(ok, 1):
        info      = r.get('info') or {}
        title     = info.get('title',     'Untitled')
        summary   = info.get('summary',   '')
        sentiment = info.get('sentiment', 'unknown')
        kp        = info.get('key_points', [])
        kp_text   = '; '.join(kp[:3]) if kp else ''
        lines.append('[' + str(i) + '] ' + title + '  [' + sentiment + ']')
        lines.append('    ' + summary)
        if kp_text:
            lines.append('    Key points: ' + kp_text)
        lines.append('')
    context  = '\n'.join(lines)
    prompt   = (
        context + '\n\n'
        'Write a 3-4 sentence editorial digest identifying '
        'the main themes, patterns, and key insights across all articles.'
    )
    response = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response['message']['content']
```

</details>